# Buổi 2 — Khoảng tin cậy & t-test với dữ liệu trường học PISA VN

In [1]:
# Chạy ô này đầu tiên. Dữ liệu (PISA 2025, Việt Nam) được tải trực tiếp từ GitHub ở ô kế tiếp — không cần tải/upload tay.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf, statsmodels.api as sm
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"]=(7,4); pd.set_option("display.precision",3)

In [2]:
# ---- TẢI DỮ LIỆU (2 bảng: cấp trường 195 dòng, cấp học sinh 7.368 dòng) ----
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
hs = pd.read_csv(RAW + "vnm_hocsinh_7368.csv")
VUNG={1:"ĐB sông Cửu Long",2:"Bắc TB, DH TB & Tây Nguyên",3:"Trung du & MN phía Bắc",4:"ĐB sông Hồng",5:"Đông Nam Bộ"}
truong["vung_ten"]=truong.vung.map(VUNG); truong["loai"]=truong.PRIVATESCH.map({1:"Công",2:"Tư"})
W=["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP","AVLRSOFT","ENCOURPG"]
print("Bảng trường:",truong.shape,"| Bảng học sinh:",hs.shape)

Bảng trường: (195, 36) | Bảng học sinh: (7368, 54)


## 1. Khoảng tin cậy 95%

In [3]:
def ci(x):
    x=x.dropna(); se=stats.sem(x); l,u=stats.t.interval(.95,len(x)-1,loc=x.mean(),scale=se); return dict(n=len(x),mean=x.mean(),sd=x.std(),se=se,lo=l,hi=u)
pd.DataFrame({"EDULEAD":ci(truong.EDULEAD),"sci_mean":ci(truong.sci_mean)}).T

,n,mean,sd,se,lo,hi
EDULEAD,195.0,0.765,0.899,0.064,0.638,0.892
sci_mean,195.0,0.446,0.090,0.006,0.433,0.459


## 2. t độc lập: trường công (161) vs tư (34) — *đọc cả hai dòng: phương sai bằng nhau và Welch*

In [4]:
def tt(v):
    a=truong[truong.PRIVATESCH==1][v]; b=truong[truong.PRIVATESCH==2][v]
    lev=stats.levene(a,b,center="mean"); pooled=stats.ttest_ind(a,b); welch=stats.ttest_ind(a,b,equal_var=False)
    sp=np.sqrt(((len(a)-1)*a.var()+(len(b)-1)*b.var())/(len(a)+len(b)-2))
    return dict(TB_cong=a.mean(),TB_tu=b.mean(),chenh=a.mean()-b.mean(),Levene_p=lev.pvalue,t_pooled=pooled.statistic,p_pooled=pooled.pvalue,t_Welch=welch.statistic,p_Welch=welch.pvalue,d=(a.mean()-b.mean())/sp)
pd.DataFrame({v:tt(v) for v in ["EDULEAD","NEGSCLIM","STAFFSHORT","sci_mean"]}).T

,TB_cong,TB_tu,chenh,Levene_p,t_pooled,p_pooled,t_Welch,p_Welch,d
EDULEAD,0.696,1.091,-0.395,0.148,-2.356,0.019,-1.874,0.068,-0.445
NEGSCLIM,-0.298,0.194,-0.492,0.530,-1.965,0.051,-1.802,0.078,-0.371
STAFFSHORT,0.142,-0.147,0.289,0.187,1.469,0.143,1.329,0.191,0.277
sci_mean,0.449,0.432,0.017,0.685,0.986,0.325,1.032,0.307,0.186


**❓** Với `EDULEAD`: p (pooled) = .019 nhưng p (Welch) = .068. Kết luận nào đáng tin hơn khi nhóm tư chỉ có 34 trường? Vì sao?

## 3. t bắt cặp & một mẫu

In [5]:
print("bắt cặp STAFFSHORT vs EDUSHORT:",stats.ttest_rel(truong.STAFFSHORT,truong.EDUSHORT))
for v in ["EDULEAD","ENCOURPG"]: print(v,"khác 0?",stats.ttest_1samp(truong[v],0))

bắt cặp STAFFSHORT vs EDUSHORT: TtestResult(statistic=np.float64(0.20183141415991795), pvalue=np.float64(0.8402600032491355), df=np.int64(194))
EDULEAD khác 0? TtestResult(statistic=np.float64(11.893894477001027), pvalue=np.float64(7.430419906525151e-25), df=np.int64(194))
ENCOURPG khác 0? TtestResult(statistic=np.float64(13.393334304345585), pvalue=np.float64(2.1491170439348348e-29), df=np.int64(194))


## 4. Trọng số trường — trung bình có trọng số khác trung bình thường

In [6]:
w=truong.W_NRASCHBWT; print("EDULEAD: không TS =",round(truong.EDULEAD.mean(),3),"| có TS =",round(np.average(truong.EDULEAD,weights=w),3))
print("Trọng số: min",round(w.min(),1),"max",round(w.max(),1),"→ một trường có thể 'đại diện' cho >600 trường")

EDULEAD: không TS = 0.765 | có TS = 0.787
Trọng số: min 1.0 max 617.2 → một trường có thể 'đại diện' cho >600 trường


## 5. Power khi hai nhóm lệch cỡ mẫu (161 vs 34)

In [7]:
from statsmodels.stats.power import TTestIndPower
p=TTestIndPower(); print("d=0.5, 161 vs 34:",round(p.power(.5,161,.05,ratio=34/161),2),"| d=0.5, 98 vs 98:",round(p.power(.5,98,.05,ratio=1),2))

d=0.5, 161 vs 34: 0.75 | d=0.5, 98 vs 98: 0.94


## Bài tập
`bai_tap/buoi2_de.md`